# Risk Analytics Operational Checks Notebook

Quick health checks for local Risk Analytics platform components:
- Spark Connect availability
- Nessie API reachability
- Iceberg catalog/table visibility
- Optional Airflow API connectivity

In [ ]:
import json
import os

import requests
from pyspark.sql import SparkSession

SPARK_REMOTE = os.getenv("SPARK_REMOTE", "sc://localhost:15002")
NESSIE_URI = os.getenv("NESSIE_URI", "http://localhost:19120/api/v2").rstrip("/")
AIRFLOW_API_URL = os.getenv("AIRFLOW_API_URL", "http://localhost:8088/api/v1").rstrip("/")
AIRFLOW_USER = os.getenv("AIRFLOW_ADMIN_USER", "admin")
AIRFLOW_PASS = os.getenv("AIRFLOW_ADMIN_PASSWORD", "admin")

print(f"SPARK_REMOTE={SPARK_REMOTE}")
print(f"NESSIE_URI={NESSIE_URI}")
print(f"AIRFLOW_API_URL={AIRFLOW_API_URL}")

In [ ]:
# 1) Nessie health and references
print("== Nessie check ==")
resp = requests.get(f"{NESSIE_URI}/trees", timeout=10)
print("status:", resp.status_code)
resp.raise_for_status()
refs = resp.json().get("references", [])
print("references:", len(refs))
for ref in refs:
    print(f"- {ref.get('type')} {ref.get('name')} @ {ref.get('hash')}")

In [ ]:
# 2) Spark Connect session + catalog visibility
print("== Spark Connect check ==")
spark = SparkSession.builder.remote(SPARK_REMOTE).appName("risk-analytics-operational-checks").getOrCreate()
print("Spark session created")
spark.sql("SHOW CATALOGS").show(truncate=False)
spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)

In [ ]:
# 3) Iceberg table checks
print("== Iceberg table checks ==")
spark.sql("SHOW TABLES IN nessie.risk_analytics").show(truncate=False)

count_df = spark.sql("SELECT COUNT(*) AS row_count FROM nessie.risk_analytics.risk_metrics")
count_df.show(truncate=False)

spark.sql("""
SELECT snapshot_id, parent_id, committed_at, operation
FROM nessie.risk_analytics.risk_metrics.snapshots
ORDER BY committed_at DESC
LIMIT 10
""").show(truncate=False)

In [ ]:
# 4) Try Nessie references via Spark SQL (if supported)
print("== Spark SQL reference listing ==")
try:
    spark.sql("SHOW REFERENCES IN nessie").show(truncate=False)
except Exception as e:
    print("Not supported in this Spark/Nessie combination.")
    print(e)

In [ ]:
# 5) Optional Airflow API check
print("== Airflow API check ==")
try:
    dag_resp = requests.get(f"{AIRFLOW_API_URL}/dags/ra_riskmetrics_eval_ods", auth=(AIRFLOW_USER, AIRFLOW_PASS), timeout=10)
    print("status:", dag_resp.status_code)
    dag_resp.raise_for_status()
    body = dag_resp.json()
    print(json.dumps({
        "dag_id": body.get("dag_id"),
        "is_paused": body.get("is_paused"),
        "next_dagrun": body.get("next_dagrun")
    }, indent=2, default=str))
except Exception as e:
    print("Airflow API check failed:", e)

In [ ]:
# Cleanup when finished
# spark.stop()